<div style="background:linear-gradient(120deg,#0B1F3A,#123A4C);color:#F6F1E7;padding:36px;border-radius:18px;">
<p style="color:#F4C95D;letter-spacing:.16em;font-size:12px;">PLAYGROUND S6E9 · BEYOND AVERAGES</p>
<h1 style="color:#F6F1E7;font-size:36px;line-height:1.15;">EV Buyers: Context Changes Everything</h1>
<p style="font-size:18px;">74 proven features. 12 conditional rates. One controlled test.</p>
<p style="color:#00B4A6;">Den Pugovkin · explain the pattern, then measure its value</p></div>


### TL;DR

Our previous notebook received **0.94625 public** (user-provided result). The lab's three-seed parent recorded **0.946061 CV / 0.94628 public**. These are historical results, **not scores for the new context block**.

This version asks a specific question: **does explicitly encoding who has access to charging help shallow trees rank buyers?** It trains the 74-feature control and 86-feature candidate on identical frozen folds, with identical seeds and stopping rules. Six fixed pairs add twelve smoothed conditional rates. No sweep, pseudo-labels or external prediction files.

**Run:** attach the competition, choose CPU, then Run All. Three seeds by default; budget roughly **1–2 hours**, an estimate rather than a measured runtime for this version. Set `SEEDS=(42,)` for a smaller five-fold paired experiment. Optional original data supports only the diagnostic comparison. Internet is unnecessary.

`submission.csv` uses the candidate only if its OOF gain exceeds **0.000020** and it wins **at least 4/5 folds**. Both standalone submissions are also saved. This is a development gate, not a guarantee about the hidden leaderboard.

[01 Research](#s1) · [02 Hidden dependencies](#s2) · [03 Conditional features](#s3) · [04 Paired training](#s4) · [05 Verdict & artifacts](#s5)


<h2 id="s1" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding:8px 0;">01 / What new idea survives the research audit?</h2>


| Source inspected | Evidence and limitation | Decision |
|---|---|---|
| [Chris Deotte: A small Simpson’s paradox](https://www.kaggle.com/competitions/playground-series-s6e9/discussion/738991) | Charging-station averages reverse after conditioning on home charging. Descriptive evidence, not a model ablation. | Test home charging × nearby stations. Recompute the chart below. |
| [Georgy Mamarin’s comment in the same discussion](https://www.kaggle.com/competitions/playground-series-s6e9/discussion/738991) | Reports a related city/home-charging composition pattern. | Include city × home charging in the fixed block. |
| [Evgeniy Dvorkin: Single XGB](https://www.kaggle.com/code/evgendvorkin/s6e9-single-xgb-cv-0-94583) | Digit, frequency and target-rate representations. | Retain the lab-tested 74-feature base. |
| [Naji: Pure LGBM](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637) | Multiscale rates and smoothing; different validation choices. | Borrow the principle of smoothing, not a transferable score claim. |
| [jazivxt: Zoom Zoom v9](https://www.kaggle.com/code/jazivxt/single-model-zoom-zoom?scriptVersionId=349430938) | Final predictions include an external reference; final public score is not pure-model evidence. | Keep the trained control and candidate self-contained. |

**Dataset decision.** [Omkar Kadam’s original table](https://www.kaggle.com/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety) is 10,000 synthetic rows, CC0 on its card. Appending its complete rows already lost all five folds in our exp025 (**0.946027 CV / 0.94621 public**). We therefore use it only to visualize distribution differences. [Zoom Zoom Baseline](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline/versions/11) is a prediction collection, not additional observed labels.

**New hypothesis:** conditional rates may make known interactions easier for shallow trees to use. No newly verified community recipe establishes that this block improves our model. Negative results remain visible. Research checked 16 September 2026.


In [ ]:
BUILD_REVISION = 'b46b3c60d58cdc696b7dab9eaa196b51e72e38b3'
NOTEBOOK_CODE_SHA = '190ce7f305b45889a70a07706bd8b02068922e0da6349b86a286a13cd0741b0e'


In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import os
import platform
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split

SEEDS = (42, 2025, 7)  # Use (42,) for the smaller paired experiment.
TARGET, ID = "Will_Buy_EV", "id"
CATS = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
        "Subsidy_Available", "Range_Anxiety_Level"]
NUMS = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
        "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
        "Environmental_Concern_Level"]
FEATURES = NUMS + CATS
EXTRA = CATS + [c for c in NUMS if c not in ("Annual_Income_USD", "Daily_Commute_km")]
NAVY, TEAL, GOLD, CORAL, PAPER = "#0B1F3A", "#00B4A6", "#F4C95D", "#E85D4C", "#F6F1E7"
STYLE = dict(paper_bgcolor=PAPER, plot_bgcolor=PAPER,
             font=dict(color=NAVY, size=13), height=440,
             margin=dict(l=55, r=25, t=65, b=50))
np.random.seed(42)


In [ ]:
def show_table(frame: pd.DataFrame, decimals: int = 5) -> None:
    styled = frame.style.format(precision=decimals).hide(axis="index")
    styled = styled.set_table_styles([
        {"selector": "th", "props": [("background", NAVY), ("color", PAPER), ("padding", "10px")]},
        {"selector": "td", "props": [("padding", "9px"), ("border-bottom", "1px solid #ddd")]},
        {"selector": "table", "props": [("width", "100%"), ("background", PAPER)]},
    ])
    display(HTML(styled.to_html()))


def data_root() -> Path:
    override = os.environ.get("S6E9_DATA_DIR")
    candidates = [Path(override)] if override else [
        Path("/kaggle/input/competitions/playground-series-s6e9"),
        Path("/kaggle/input/playground-series-s6e9"),
        Path("data/raw"), Path("../../data/raw"),
        Path("competitions/predicting-electric-vehicle-purchases/data/raw"),
    ]
    for root in candidates:
        if all((root / name).is_file() for name in ("train.csv", "test.csv", "sample_submission.csv")):
            return root
    raise FileNotFoundError("Attach Playground S6E9, or set S6E9_DATA_DIR.")


DATA = data_root()
OUTPUT = Path(os.environ.get("S6E9_OUTPUT_DIR", "/kaggle/working" if Path("/kaggle/working").exists() else "outputs/context-story"))
OUTPUT.mkdir(parents=True, exist_ok=True)
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")


In [ ]:
assert set(train.columns) == {ID, TARGET, *FEATURES}
assert set(test.columns) == {ID, *FEATURES}
assert list(sample.columns) == [ID, TARGET]
assert train[ID].is_unique and test[ID].is_unique and sample[ID].is_unique
assert not train[ID].isin(test[ID]).any()
assert set(sample[ID]) == set(test[ID])
assert set(train[TARGET].unique()) == {"Yes", "No"}
assert not train.isna().any().any() and not test.isna().any().any()
assert np.isfinite(train[NUMS].to_numpy()).all()
assert np.isfinite(test[NUMS].to_numpy()).all()
y = train[TARGET].map({"No": 0, "Yes": 1}).astype(int)
show_table(pd.DataFrame({"Table": ["Train", "Test"],
    "Rows": [len(train), len(test)], "Features": [len(FEATURES)] * 2,
    "Missing cells": [int(train.isna().sum().sum()), int(test.isna().sum().sum())]}), 0)
print(f"Positive prevalence: {y.mean():.3%}; metric: ROC AUC, higher is better.")


In [ ]:
original_override = os.environ.get("S6E9_ORIGINAL_CSV")
original_paths = [Path(original_override)] if original_override else sorted(
    Path("/kaggle/input").rglob("EV_Adoption_and_Range_Anxiety_Dataset.csv"))
assert len(original_paths) <= 1, "Multiple original inputs: set S6E9_ORIGINAL_CSV explicitly."
original = pd.read_csv(original_paths[0]) if original_paths else None
if original is not None:
    assert set(FEATURES + [TARGET]).issubset(original.columns)
    assert original[TARGET].dropna().isin(["Yes", "No"]).all()
    print(f"Original diagnostic: {len(original):,} rows; never appended to training.")
else:
    print("Original data not attached: comparison is skipped; training is unchanged.")


<h2 id="s2" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding:8px 0;">02 / Why an overall average can point the wrong way</h2>


**Question:** do more nearby chargers mean a higher purchase rate?

We first inspect the aggregate and then condition on home charging. Each point includes its group size. A reversal reflects a different mixture of buyers; it is not proof of a causal effect. These charts use training labels for exploration only; feature encoders below learn their statistics strictly inside training partitions.


In [ ]:
station_band = pd.cut(train["Charging_Stations_Near_Home"],
                      [-np.inf, 2, 5, 10, np.inf], labels=["0–2", "3–5", "6–10", "11+"])
probe = pd.DataFrame({"Nearby stations": station_band, "Home charging": train["Home_Charging_Possible"], "yes": y})
conditional = probe.groupby(["Nearby stations", "Home charging"], observed=True).yes.agg(["mean", "size"]).reset_index()
aggregate = probe.groupby("Nearby stations", observed=True).yes.agg(["mean", "size"]).reset_index()
aggregate["Home charging"] = "All buyers"
charging_rates = pd.concat([aggregate, conditional], ignore_index=True)
fig = px.line(charging_rates, x="Nearby stations", y="mean", color="Home charging",
              markers=True, hover_data=["size"], title="Same stations, different context",
              color_discrete_map={"All buyers": NAVY, "Yes": TEAL, "No": CORAL},
              labels={"mean": "Observed purchase rate"})
fig.update_layout(**STYLE)
fig.update_yaxes(tickformat=".0%")
fig.show()
show_table(charging_rates.rename(columns={"size": "Rows", "mean": "Purchase rate"}))


**Interpretation:** the aggregate includes a changing share of people who can charge at home. Conditional curves ask a more precise question: how does the relationship look among people with the same home-charging access?

We also inspect subsidy × anxiety and compare it with the optional original table. This motivates two further contexts: concern within subsidy status, and commute length within anxiety level. Those extensions are hypotheses, not discoveries with measured model gains.


In [ ]:
def switch_summary(frame: pd.DataFrame, source: str) -> pd.DataFrame:
    valid = frame.dropna(subset=[TARGET, "Subsidy_Available", "Range_Anxiety_Level"])
    valid = valid.assign(yes=valid[TARGET].map({"No": 0, "Yes": 1}))
    result = valid.groupby(["Subsidy_Available", "Range_Anxiety_Level"], observed=True)["yes"].agg(["mean", "size"]).reset_index()
    return result.assign(source=source)


switches = switch_summary(train, "Playground")
if original is not None:
    switches = pd.concat([switches, switch_summary(original, "Original")], ignore_index=True)
fig = px.bar(switches, x="Range_Anxiety_Level", y="mean", color="Subsidy_Available",
             facet_col="source", barmode="group", hover_data=["size"],
             color_discrete_map={"No": CORAL, "Yes": TEAL},
             category_orders={"Range_Anxiety_Level": ["Low", "Medium", "High"]},
             labels={"mean": "Observed purchase rate"},
             title="The purchase rate depends on both subsidy and anxiety")
fig.update_layout(**STYLE)
fig.update_yaxes(tickformat=".0%")
fig.show()
show_table(switches.rename(columns={"mean": "Purchase rate", "size": "Rows"}))


<h2 id="s3" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding:8px 0;">03 / Turn context into 12 usable features</h2>


The control retains **13 raw + 16 digit + 13 frequency + 32 nested target-rate = 74 features**. Digits expose repeated synthetic numeric structure, frequencies distinguish common values, and target rates summarize purchase tendencies. These blocks have historical ablation support in the [experiment ledger](https://github.com/mart1ny/Kaggle_prof/tree/main/competitions/predicting-electric-vehicle-purchases/experiments).

The candidate adds **six fixed pairs × two smoothing strengths = 12 features**, for **86 total**:

| Parent context | Child detail | What the feature expresses |
|---|---|---|
| Home charging | Stations near home | Public charging availability within home-access status |
| Home charging | City type | City differences within a comparable access group |
| Subsidy | Range anxiety | Anxiety-related purchase tendency within subsidy status |
| Subsidy | Environmental concern | Concern-related purchase tendency within subsidy status |
| Range anxiety | Fixed 10-km commute bin | Commute-related tendency within anxiety level |
| Home charging | Stations near work | Work infrastructure within home-access status |

<div style="background:#F6F1E7;border-left:5px solid #00B4A6;padding:18px;color:#0B1F3A;">
<b>Parent rate = (parent positives + 100 × fit prior) / (parent count + 100)</b><br>
<b>Pair rate = (pair positives + m × parent rate) / (pair count + m)</b><br>
m ∈ {100, 1000}. Unseen pair → parent rate. Unseen parent → fitting-partition prior.</div>

For example, a pair with 2 positives in 10 rows and parent rate 0.10 becomes **0.1091** with m=100, rather than a noisy 0.20. Stronger smoothing pulls it closer to its parent. The exact parent rate itself is learned from the fitting partition.

**Why this might help:** shallow trees receive the conditional tendency in one numerical column instead of rediscovering every conjunction through several splits. **Why it might fail:** trees may already capture the interaction; rates can be redundant or noisy. Only the paired experiment decides.

**Isolation:** five target-independent inner KFold blocks exclude the entire held block from joint counts, parent counts and global prior. Validation and test rows contribute no learned statistics. Early stopping repeats this isolation inside each outer fold.


In [ ]:
def digit_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Deterministic transforms only; no ID, labels or learned thresholds."""
    income = frame["Annual_Income_USD"].to_numpy(dtype=float)
    commute = frame["Daily_Commute_km"].to_numpy(dtype=float)
    if not np.isfinite(income).all() or not np.isfinite(commute).all():
        raise ValueError("Non-finite income or commute")
    if not np.allclose(income, np.rint(income), rtol=0, atol=1e-6):
        raise ValueError("Income no longer satisfies the integer data contract")
    i = np.rint(income).astype(np.int64)
    tenths = np.rint(commute * 10).astype(np.int64)
    out = pd.DataFrame(index=frame.index)
    for power in (1, 10, 100, 1000):
        out[f"income_digit_{power}"] = (i // power) % 10
    for modulus in (10, 100, 1000):
        out[f"income_mod_{modulus}"] = i % modulus
        out[f"income_round_{modulus}"] = (i % modulus == 0).astype(np.int8)
    out["commute_tenth"] = tenths % 10
    out["commute_integer"] = np.floor(commute)
    out["commute_fraction"] = commute - np.floor(commute)
    out["commute_mod_5"] = tenths % 50
    out["commute_mod_10"] = tenths % 100
    out["commute_is_integer"] = np.isclose(
        commute, np.rint(commute), rtol=0, atol=1e-6
    ).astype(np.int8)
    return out


In [ ]:
def target_keys(raw: pd.DataFrame) -> pd.DataFrame:
    income = raw["Annual_Income_USD"].astype(np.float64)
    commute = raw["Daily_Commute_km"].astype(np.float64)
    keys = pd.DataFrame({"income_exact": income,
        "income_floor100": np.floor(income / 100),
        "income_floor1000": np.floor(income / 1000),
        "commute_exact": commute, "commute_integer": np.floor(commute)}, index=raw.index).astype(str)
    for col in EXTRA:
        keys[col] = raw[col].astype(str)
    return keys


def rate_maps(keys: pd.DataFrame, labels: pd.Series) -> tuple[float, dict]:
    assert keys.index.equals(labels.index) and labels.isin([0, 1]).all()
    prior, maps = float(labels.mean()), {}
    for col in keys:
        stats = pd.DataFrame({"key": keys[col], "y": labels}).groupby("key").y.agg(["sum", "count"])
        for smoothing in (10.0, 100.0):
            name = f"target_rate_{col}_s{smoothing:g}"
            maps[name] = (col, (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing))
    return prior, maps


def apply_rates(keys: pd.DataFrame, prior: float, maps: dict) -> pd.DataFrame:
    return pd.DataFrame({name: keys[col].map(mapping).fillna(prior)
                         for name, (col, mapping) in maps.items()}, index=keys.index).astype(np.float32)


In [ ]:
def crossfit_rates(raw: pd.DataFrame, labels: pd.Series) -> pd.DataFrame:
    keys = target_keys(raw)
    out = np.full((len(raw), 32), np.nan, dtype=np.float32)
    counts = np.zeros(len(raw), dtype=np.int8)
    columns = None
    for fit, held in KFold(5, shuffle=True, random_state=42).split(raw):
        prior, maps = rate_maps(keys.iloc[fit], labels.iloc[fit])
        block = apply_rates(keys.iloc[held], prior, maps)
        out[held], columns = block.to_numpy(), block.columns
        counts[held] += 1
    assert (counts == 1).all() and np.isfinite(out).all()
    return pd.DataFrame(out, index=raw.index, columns=columns)


def base_features(fit: pd.DataFrame, other: pd.DataFrame) -> pd.DataFrame:
    out = other[FEATURES].copy()
    for col in CATS:
        out[col] = pd.Categorical(other[col].astype(str), categories=sorted(fit[col].astype(str).unique()))
    out = pd.concat([out, digit_features(other)], axis=1)
    for col in FEATURES:
        freq = fit[col].astype(str).value_counts(normalize=True)
        out[f"freq_{col}"] = other[col].astype(str).map(freq).fillna(0).astype(np.float32)
    return out


def prepare_base_features(fit: pd.DataFrame, labels: pd.Series, *others: pd.DataFrame) -> list[pd.DataFrame]:
    prior, maps = rate_maps(target_keys(fit), labels)
    result = [pd.concat([base_features(fit, fit), crossfit_rates(fit, labels)], axis=1)]
    for other in others:
        result.append(pd.concat([base_features(fit, other), apply_rates(target_keys(other), prior, maps)], axis=1))
    for frame in result:
        assert frame.shape[1] == 74 and ID not in frame and TARGET not in frame
        assert np.isfinite(frame.select_dtypes(include="number").to_numpy()).all()
    return result


In [ ]:
CONTEXT_PAIRS = (
    ("Home_Charging_Possible", "Charging_Stations_Near_Home"),
    ("Home_Charging_Possible", "City_Type"),
    ("Subsidy_Available", "Range_Anxiety_Level"),
    ("Subsidy_Available", "Environmental_Concern_Level"),
    ("Range_Anxiety_Level", "commute_bin10"),
    ("Home_Charging_Possible", "Charging_Stations_Near_Work"),
)


In [ ]:
CONTEXT_SMOOTHINGS = (100.0, 1000.0)


In [ ]:
PARENT_SMOOTHING = 100.0


In [ ]:
def context_keys(raw: pd.DataFrame) -> pd.DataFrame:
    """Construct fixed keys without targets, row IDs, or learned thresholds."""
    columns = sorted({c for pair in CONTEXT_PAIRS for c in pair} - {"commute_bin10"})
    assert not raw[[*columns, "Daily_Commute_km"]].isna().any().any()
    keys = raw[columns].astype(str).copy()
    keys["commute_bin10"] = np.floor(raw["Daily_Commute_km"] / 10).astype(str)
    return keys


In [ ]:
def fit_context(keys: pd.DataFrame, labels: pd.Series) -> tuple[float, dict]:
    """Learn parent and pair counts using only the supplied fitting partition."""
    assert keys.index.is_unique and keys.index.equals(labels.index)
    assert len(labels) and labels.isin([0, 1]).all()
    prior = float(labels.mean())
    maps = {}
    for parent, child in CONTEXT_PAIRS:
        data = keys[[parent, child]].assign(_target=labels)
        p = data.groupby(parent, observed=True)._target.agg(["sum", "count"])
        parent_rate = (p["sum"] + PARENT_SMOOTHING * prior) / (
            p["count"] + PARENT_SMOOTHING
        )
        joint = data.groupby([parent, child], observed=True)._target.agg(
            ["sum", "count"]
        )
        backoff = joint.index.get_level_values(0).map(parent_rate).to_numpy()
        for smoothing in CONTEXT_SMOOTHINGS:
            rate = (joint["sum"] + smoothing * backoff) / (joint["count"] + smoothing)
            maps[f"context_{parent}__{child}_s{smoothing:g}"] = (
                parent,
                child,
                parent_rate,
                rate,
            )
    return prior, maps


In [ ]:
def apply_context(keys: pd.DataFrame, prior: float, maps: dict) -> pd.DataFrame:
    """Back off unseen pairs to their parent, then unseen parents to fit prior."""
    result = {}
    for name, (parent, child, parent_rate, rate) in maps.items():
        index = pd.MultiIndex.from_frame(keys[[parent, child]])
        joint = rate.reindex(index).to_numpy()
        backoff = keys[parent].map(parent_rate).fillna(prior).to_numpy()
        result[name] = np.where(np.isnan(joint), backoff, joint)
    out = pd.DataFrame(result, index=keys.index, dtype=np.float32)
    assert np.isfinite(out.to_numpy()).all()
    assert ((out >= 0) & (out <= 1)).all().all()
    return out


In [ ]:
def crossfit_context(raw: pd.DataFrame, labels: pd.Series) -> pd.DataFrame:
    """Exclude the entire held block from both levels of the hierarchical prior."""
    keys = context_keys(raw)
    assert len(raw) >= 5 and raw.index.equals(labels.index)
    out = np.full((len(raw), len(CONTEXT_PAIRS) * len(CONTEXT_SMOOTHINGS)), np.nan)
    seen = np.zeros(len(raw), dtype=np.int8)
    columns = None
    for fit, held in KFold(5, shuffle=True, random_state=42).split(raw):
        prior, maps = fit_context(keys.iloc[fit], labels.iloc[fit])
        block = apply_context(keys.iloc[held], prior, maps)
        out[held], columns = block.to_numpy(), block.columns
        seen[held] += 1
    assert (seen == 1).all() and np.isfinite(out).all()
    return pd.DataFrame(out, index=raw.index, columns=columns, dtype=np.float32)


In [ ]:
def paired_decision(
    labels: pd.Series,
    baseline: np.ndarray,
    candidate: np.ndarray,
    folds: np.ndarray,
    min_delta: float = 0.00002,
) -> dict:
    """A predeclared development gate; its selected CV is not an unbiased estimate."""
    from sklearn.metrics import roc_auc_score

    assert len(labels) == len(baseline) == len(candidate) == len(folds)
    assert set(folds) == set(range(5)) and min_delta >= 0
    for prediction in (baseline, candidate):
        assert np.isfinite(prediction).all()
        assert ((prediction >= 0) & (prediction <= 1)).all()
    scores = {}
    for name, prediction in (("baseline", baseline), ("context", candidate)):
        scores[name] = {
            "auc": float(roc_auc_score(labels, prediction)),
            "per_fold": [
                float(roc_auc_score(labels[folds == f], prediction[folds == f]))
                for f in range(5)
            ],
        }
    delta = scores["context"]["auc"] - scores["baseline"]["auc"]
    deltas = np.subtract(scores["context"]["per_fold"], scores["baseline"]["per_fold"])
    wins = int((deltas > 0).sum())
    return {
        "scores": scores,
        "delta": delta,
        "fold_deltas": deltas.tolist(),
        "fold_wins": wins,
        "minimum_delta": min_delta,
        "selected": "context" if delta > min_delta and wins >= 4 else "baseline",
        "selection_note": "Development-selected CV; independent holdout not evaluated.",
    }


In [ ]:
def prepare_features(fit: pd.DataFrame, labels: pd.Series,
                     *others: pd.DataFrame, recipe: str = "baseline") -> list[pd.DataFrame]:
    assert recipe in ("baseline", "context")
    result = prepare_base_features(fit, labels, *others)
    if recipe == "context":
        prior, maps = fit_context(context_keys(fit), labels)
        blocks = [crossfit_context(fit, labels)]
        blocks.extend(apply_context(context_keys(other), prior, maps) for other in others)
        result = [pd.concat([frame, block], axis=1) for frame, block in zip(result, blocks, strict=True)]
    for frame in result:
        assert frame.shape[1] == (86 if recipe == "context" else 74)
        assert ID not in frame and TARGET not in frame and frame.columns.is_unique
        assert np.isfinite(frame.select_dtypes(include="number").to_numpy()).all()
    return result


<h2 id="s4" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding:8px 0;">04 / Same rows, same seeds, one material change</h2>


**Metric:** ROC AUC on P(Yes), higher is better. **Fold version:** canonical v1, five folds, seed 42, fingerprint verified below. The split is explicitly restored only if no supplied fold artifact exists; it must match the historical fingerprint or execution stops.

**Control:** exp019 three-seed 74-feature bag (exp015 when using only seed 42). **Candidate:** exactly the same recipe plus the twelve context rates. Each model chooses tree count on an inner 10% stopping split, then refits freshly on the full outer training partition. Outer labels never select tree count.

We save each completed seed/fold prediction file immediately. These are diagnostic checkpoints, not automatic resume files. Final OOF has exactly one averaged prediction per training ID. Changing seeds changes the experiment, so compare the two recipes from this same run.


In [ ]:
FOLD_SHA = "f50fdbcd462ab8ae7ad189688f7ba1e0876230b639e80af0218afb042fc4fe7b"


def frozen_folds(frame: pd.DataFrame, labels: pd.Series) -> pd.DataFrame:
    supplied = os.environ.get("S6E9_FOLDS_PATH")
    if supplied:
        assignment = pd.read_parquet(supplied)
    else:
        print("Explicitly restoring frozen v1; the canonical fingerprint MUST match.")
        fold_ids = np.full(len(frame), -1, dtype=np.int64)
        for fold, (_, held) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(frame, labels)):
            fold_ids[held] = fold
        assignment = pd.DataFrame({ID: frame[ID], "fold": fold_ids})
    assert list(assignment.columns) == [ID, "fold"] and assignment[ID].is_unique
    assert set(assignment[ID]) == set(frame[ID]) and set(assignment.fold) == set(range(5))
    ordered = assignment.sort_values(ID)
    digest = hashlib.sha256(ordered.to_numpy(dtype="<i8").tobytes()).hexdigest()
    assert digest == FOLD_SHA, "Frozen fold mismatch: stop; do not regenerate a new split."
    aligned = assignment.set_index(ID).loc[frame[ID]].reset_index()
    aligned.to_parquet(OUTPUT / "folds_v1.parquet", index=False)
    return aligned


fold_table = frozen_folds(train, y)
show_table(fold_table.groupby("fold").size().rename("Rows").reset_index(), 0)


In [ ]:
PARAMS = {'n_estimators': 3500, 'learning_rate': 0.02, 'max_depth': 5, 'num_leaves': 32, 'min_child_samples': 10, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.3, 'reg_alpha': 0.071, 'reg_lambda': 2.0, 'max_bin': 255, 'feature_pre_filter': False, 'random_state': 42, 'n_jobs': 4, 'verbose': -1, 'metric': 'auc'}
assert SEEDS and len(set(SEEDS)) == len(SEEDS)


In [ ]:
def fit_one_fold(raw: pd.DataFrame, labels: pd.Series,
                 valid: pd.DataFrame, infer: pd.DataFrame, seed: int, recipe: str) -> tuple:
    inner, stop = train_test_split(np.arange(len(raw)), test_size=0.1,
                                  stratify=labels, random_state=42)
    x_inner, x_stop = prepare_features(raw.iloc[inner], labels.iloc[inner], raw.iloc[stop], recipe=recipe)
    params = dict(PARAMS, random_state=seed)
    selector = lgb.LGBMClassifier(**params)
    selector.fit(x_inner, labels.iloc[inner], eval_set=[(x_stop, labels.iloc[stop])],
                 eval_metric="auc", callbacks=[lgb.early_stopping(100, first_metric_only=True), lgb.log_evaluation(0)])
    trees = int(selector.best_iteration_)
    assert 1 <= trees <= params["n_estimators"]
    del selector, x_inner, x_stop
    x_full, x_valid, x_test = prepare_features(raw, labels, valid, infer, recipe=recipe)
    model = lgb.LGBMClassifier(**dict(params, n_estimators=trees))
    model.fit(x_full, labels)
    valid_pred = model.predict_proba(x_valid)[:, 1]
    test_pred = model.predict_proba(x_test)[:, 1]
    importance = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=x_full.columns)
    return valid_pred, test_pred, trees, importance


In [ ]:
def train_cv(raw: pd.DataFrame, labels: pd.Series, infer: pd.DataFrame,
             assignment: pd.DataFrame, recipe: str) -> tuple:
    assert assignment[ID].equals(raw[ID])
    oof_sum, test_sum = np.zeros(len(raw)), np.zeros(len(infer))
    rows, importances = [], []
    for seed in SEEDS:
        seen = np.zeros(len(raw), dtype=np.int8)
        for fold in range(5):
            started = time.perf_counter()
            fit = np.flatnonzero(assignment.fold.to_numpy() != fold)
            held = np.flatnonzero(assignment.fold.to_numpy() == fold)
            pred, pred_test, trees, imp = fit_one_fold(raw.iloc[fit], labels.iloc[fit], raw.iloc[held], infer, seed, recipe)
            assert np.isfinite(pred).all() and np.isfinite(pred_test).all()
            assert ((pred >= 0) & (pred <= 1)).all()
            assert ((pred_test >= 0) & (pred_test <= 1)).all()
            oof_sum[held] += pred / len(SEEDS)
            test_sum += pred_test / (5 * len(SEEDS))
            seen[held] += 1
            checkpoint = OUTPUT / "checkpoints"
            checkpoint.mkdir(exist_ok=True)
            pd.DataFrame({ID: raw.iloc[held][ID], "prediction": pred}).to_parquet(checkpoint / f"{recipe}_{seed}_{fold}_valid.parquet", index=False)
            pd.DataFrame({ID: infer[ID], "prediction": pred_test}).to_parquet(checkpoint / f"{recipe}_{seed}_{fold}_test.parquet", index=False)
            row = dict(seed=seed, fold=fold, auc=float(roc_auc_score(labels.iloc[held], pred)),
                       trees=trees, seconds=time.perf_counter() - started)
            rows.append(row)
            importances.append(imp)
            print(row, flush=True)
            gc.collect()
        assert (seen == 1).all(), "Each row needs exactly one held-out prediction per seed."
    return oof_sum, test_sum, pd.DataFrame(rows), pd.concat(importances, axis=1).mean(axis=1)


In [ ]:
started = time.perf_counter()
results = {}
for recipe in ("baseline", "context"):
    print(f"\nTRAINING {recipe.upper()} — {len(SEEDS)} seeds × 5 frozen folds", flush=True)
    oof_run, test_run, scores_run, gain_run = train_cv(train, y, test, fold_table, recipe)
    results[recipe] = dict(oof=oof_run, test=test_run, scores=scores_run, gain=gain_run)
    pd.DataFrame({ID: train[ID], "fold": fold_table.fold, "prediction": oof_run}).to_parquet(OUTPUT / f"oof_{recipe}.parquet", index=False)
    pd.DataFrame({ID: test[ID], TARGET: test_run}).to_parquet(OUTPUT / f"test_{recipe}.parquet", index=False)
    scores_run.to_csv(OUTPUT / f"fold_scores_{recipe}.csv", index=False)
    print(f"{recipe}: OOF AUC = {roc_auc_score(y, oof_run):.9f}", flush=True)
duration = time.perf_counter() - started
verdict = paired_decision(y, results["baseline"]["oof"], results["context"]["oof"], fold_table.fold.to_numpy())
selected = verdict["selected"]
print(json.dumps(verdict, indent=2))


<h2 id="s5" style="color:#0B1F3A;border-bottom:3px solid #00B4A6;padding:8px 0;">05 / Did context earn its place?</h2>


The following panels are computed from **this run**. Fold gains show consistency; the overall metric also includes rankings between observations in different folds and need not equal their average. Five fold wins are not five independent replications because training partitions overlap.

The fixed gate prevents automatically choosing every tiny positive fluctuation. It does **not** remove selection bias: the selected CV is a development result, and repeated experiments on v1 can overfit it. We do not estimate a new public score from CV.


In [ ]:
comparison = pd.DataFrame({"Fold": range(5),
    "Baseline AUC": verdict["scores"]["baseline"]["per_fold"],
    "Context AUC": verdict["scores"]["context"]["per_fold"],
    "Delta": verdict["fold_deltas"]})
show_table(comparison, 7)
fig = px.bar(comparison, x="Fold", y="Delta", color=comparison.Delta.gt(0),
             color_discrete_map={True: TEAL, False: CORAL},
             title="Paired fold gains: context minus the identical control")
fig.update_layout(**STYLE, showlegend=False)
fig.add_hline(y=0, line_color=NAVY)
fig.show()
decision_color = TEAL if selected == "context" else CORAL
display(HTML(f'<div style="background:{NAVY};color:{PAPER};padding:22px;border-radius:16px;">'
    f'<h3 style="color:{decision_color};">Selected: {selected.upper()}</h3>'
    f'OOF delta: {verdict["delta"]:+.7f} · Fold wins: {verdict["fold_wins"]}/5<br>'
    'Gate: delta &gt; 0.000020 and at least four positive fold deltas.<br>'
    'Public leaderboard result: not measured for this run.</div>'))


In [ ]:
slice_rows = []
for column in ("Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"):
    for value in sorted(train[column].unique()):
        mask = train[column].eq(value).to_numpy()
        labels = y[mask]
        row = {"Slice": f"{column}={value}", "Rows": int(mask.sum()), "Positives": int(labels.sum())}
        for recipe in results:
            row[recipe] = float(roc_auc_score(labels, results[recipe]["oof"][mask])) if labels.nunique() == 2 else np.nan
        row["Delta"] = row["context"] - row["baseline"]
        slice_rows.append(row)
slices = pd.DataFrame(slice_rows)
show_table(slices, 7)
gain = results["context"]["gain"]
context_gain = gain[gain.index.str.startswith("context_")].sort_values().rename_axis("Feature").reset_index(name="Gain")
short_names = {"context_": "", "Home_Charging_Possible": "Home access", "Charging_Stations_Near_Home": "Home stations",
               "Charging_Stations_Near_Work": "Work stations", "Subsidy_Available": "Subsidy", "Range_Anxiety_Level": "Anxiety",
               "Environmental_Concern_Level": "Concern", "City_Type": "City", "commute_bin10": "Commute 10km", "__": " × ", "_s": " · m="}
for full, short in short_names.items():
    context_gain["Feature"] = context_gain["Feature"].str.replace(full, short, regex=False)
fig = px.bar(context_gain, x="Gain", y="Feature", orientation="h", color_discrete_sequence=[TEAL],
             title="Which conditional rates were used? Gain is not causal importance")
fig.update_layout(**dict(STYLE, height=560, margin=dict(l=40, r=20, t=65, b=50)))
fig.show()


**How to read the outcome.** If the gate passes, the whole conditional block improved this paired development comparison. It does not prove every individual pair helped. Slice results describe where ranking changed; small positive counts make some slices unstable. Split gain tells us what the model used, not the effect of deleting a feature.

If the gate fails, the notebook keeps the control and records the negative experiment. Context may explain the data well without providing additional predictive signal. That is a useful result.

**Our own dataset:** this run creates ID-preserving OOF predictions for both recipes and a feature dictionary. These enable future paired comparisons without rerunning the control. They are derived predictions, not new ground-truth labels. Keep row-level competition artifacts private unless redistribution is permitted by the competition rules.


In [ ]:
def file_sha(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


for recipe, result in results.items():
    predictions = pd.DataFrame({ID: test[ID], TARGET: result["test"]})
    submission = sample[[ID]].merge(predictions, on=ID, how="left", validate="one_to_one")
    assert submission[ID].equals(sample[ID]) and list(submission.columns) == [ID, TARGET]
    assert np.isfinite(submission[TARGET]).all() and submission[TARGET].between(0, 1).all()
    submission.to_csv(OUTPUT / f"submission_{recipe}.csv", index=False)
    if recipe == selected:
        submission.to_csv(OUTPUT / "submission.csv", index=False)
comparison.to_csv(OUTPUT / "paired_fold_comparison.csv", index=False)
slices.to_csv(OUTPUT / "slice_metrics.csv", index=False)
gain.rename_axis("feature").rename("gain").to_csv(OUTPUT / "feature_importance_context.csv")


In [ ]:
def feature_description(name: str) -> tuple[str, str, str]:
    if name.startswith("context_"):
        return "conditional target rate", "Smoothed pair rate backed off to parent context", "Both levels cross-fitted; unseen parent=fit prior"
    if name in FEATURES:
        return "raw", "Original measurement or category", "Training vocabulary for categories"
    if name.startswith("freq_"):
        return "frequency", "Relative count of the source value", "Fit partition only; unseen=0"
    if name.startswith("target_rate_"):
        return "target rate", "Smoothed purchase rate for key and smoothing in name", "Cross-fitted train; fit-only validation/test maps"
    return "digits", "Deterministic digit, remainder or rounding transform", "No learned statistics"


feature_dictionary = pd.DataFrame([
    {"feature": name, "block": feature_description(name)[0],
     "meaning": feature_description(name)[1], "fit_scope": feature_description(name)[2]}
    for name in gain.index])
feature_dictionary.to_csv(OUTPUT / "feature_dictionary.csv", index=False)
bag_folds = [{"fold": f, "auc": float(roc_auc_score(y[fold_table.fold.eq(f)], results[selected]['oof'][fold_table.fold.eq(f)]))} for f in range(5)]
metrics = {"status": "completed", "metric": "roc_auc", "direction": "maximize", "cv": verdict["scores"][selected]["auc"],
           "per_fold": bag_folds, "seeds": list(SEEDS), "duration_seconds": duration,
           "fold_version": "v1", "fold_sha256": FOLD_SHA, "parent": "exp019" if len(SEEDS) == 3 else "exp015",
           "feature_version": "context_rates_v1" if selected == "context" else "synthetic_v1_plus_extra_target_v1", "feature_count": 86 if selected == "context" else 74,
           "paired_experiment": verdict, "selected_recipe": selected,
           "submission_sha256": file_sha(OUTPUT / "submission.csv"), "public_lb": None}
(OUTPUT / "metrics.json").write_text(json.dumps(metrics, indent=2))


In [ ]:
artifact_names = ["submission.csv", "submission_baseline.csv", "submission_context.csv",
    "oof_baseline.parquet", "oof_context.parquet", "test_baseline.parquet", "test_context.parquet",
    "folds_v1.parquet", "fold_scores_baseline.csv", "fold_scores_context.csv",
    "paired_fold_comparison.csv", "slice_metrics.csv", "feature_dictionary.csv", "metrics.json"]
manifest = {"recipe": PARAMS, "seeds": list(SEEDS), "fold_seed": 42, "inner_seed": 42,
    "target_encoding": {"splits": 5, "seed": 42, "smoothings": [10, 100]},
    "context": {"pairs": CONTEXT_PAIRS, "smoothings": CONTEXT_SMOOTHINGS, "parent_smoothing": PARENT_SMOOTHING},
    "gate": {"minimum_delta": 0.00002, "minimum_fold_wins": 4},
    "environment": {name: importlib.metadata.version(name) for name in
                    ["numpy", "pandas", "scikit-learn", "lightgbm", "plotly"]},
    "python": platform.python_version(), "platform": platform.platform(),
    "source_revision": BUILD_REVISION, "notebook_code_sha256": NOTEBOOK_CODE_SHA,
    "input_sha256": {name: file_sha(DATA / name) for name in ["train.csv", "test.csv", "sample_submission.csv"]},
    "original_used_for_training": False,
    "original_sha256": file_sha(original_paths[0]) if original_paths else None,
    "artifacts": {name: file_sha(OUTPUT / name) for name in artifact_names}}
(OUTPUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("READY:", OUTPUT / "submission.csv", "| selected recipe:", selected)
print("SHA-256:", metrics["submission_sha256"])


### What to test next, after reading the verdict

1. **If the block passes:** remove one semantically related subgroup at a time to locate its contribution. Moderate compute; repeated-selection risk.
2. **If it fails:** retain the 74-feature control and inspect paired slice errors before proposing another block. No additional training needed for the diagnosis.
3. **Only with new transfer evidence:** test original-data priors after a duplicate/overlap audit. Do not repeat the failed row-augmentation experiment unchanged.

Download `submission.csv`, `metrics.json`, `manifest.json`, and the paired OOF files. Record the actual public score only after submission. Source ideas and datasets are credited in section 01; code is an independently implemented extension of our tested baseline.

<div style="background:#0B1F3A;color:#F6F1E7;border-radius:16px;padding:22px;">
<b>A stronger notebook makes the improvement testable.</b><br>
Historical scores stay historical. New results come from the run.<br>
Code and original narrative: MIT · Den Pugovkin, 2026.</div>
